# Project 01 — Titanic Exploratory Data Analysis

**Goal:** inspect, clean, and summarize the Titanic passenger dataset with Pandas.

**Dataset:** public seaborn-data Titanic sample (891 passenger rows). Source: [seaborn-data](https://github.com/mwaskom/seaborn-data/blob/master/titanic.csv).

This notebook keeps exact duplicate-looking rows because the public file has no passenger identifier with which to prove that they are duplicate people. It reports the count and avoids silently deleting records.

In [1]:
from pathlib import Path
import warnings
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
candidate_paths = [Path("../data/titanic.csv"), Path("data/titanic.csv"), Path("../../data/titanic.csv")]
DATA_PATH = next((path for path in candidate_paths if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Place the included titanic.csv in the portfolio data folder.")
raw = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {raw.shape[0]} rows × {raw.shape[1]} columns")
display(raw.head())

Dataset shape: 891 rows × 15 columns
   survived  pclass     sex   age  ...  deck  embark_town  alive  alone
0         0       3    male  22.0  ...   NaN  Southampton     no  False
1         1       1  female  38.0  ...     C    Cherbourg    yes  False
2         1       3  female  26.0  ...   NaN  Southampton    yes   True
3         1       1  female  35.0  ...     C  Southampton    yes  False
4         0       3    male  35.0  ...   NaN  Southampton     no   True

[5 rows x 15 columns]


## 1. Inspect the data

In [2]:
raw.info()
display(raw.describe(include="all").T)
missing = raw.isna().sum().sort_values(ascending=False)
display(missing[missing.gt(0)].rename("missing_rows").to_frame())
exact_duplicate_profiles = int(raw.duplicated().sum())
print(f"Exact duplicate-looking rows: {exact_duplicate_profiles}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   survived     891 non-null    int64  
 1   pclass       891 non-null    int64  
 2   sex          891 non-null    object 
 3   age          714 non-null    float64
 4   sibsp        891 non-null    int64  
 5   parch        891 non-null    int64  
 6   fare         891 non-null    float64
 7   embarked     889 non-null    object 
 8   class        891 non-null    object 
 9   who          891 non-null    object 
 10  adult_male   891 non-null    bool   
 11  deck         203 non-null    object 
 12  embark_town  889 non-null    object 
 13  alive        891 non-null    object 
 14  alone        891 non-null    bool   
dtypes: bool(2), float64(2), int64(4), object(7)
memory usage: 92.4+ KB
             count unique          top freq  ...     25%      50%   75%       max
survived     891.0    NaN  

## 2. Clean missing values carefully

- `deck` is omitted because most entries are blank and the notebook does not need it for these questions.
- Missing ages are filled with the median within passenger class and sex, with the overall median as a fallback.
- Missing embarkation ports use the most common recorded port.
- Duplicate-looking rows are retained: without a passenger ID, identical measured attributes do not prove the same person was recorded twice.

In [3]:
clean = raw.drop(columns=["deck"]).copy()
overall_age_median = clean["age"].median()
clean["age"] = clean.groupby(["pclass", "sex"])["age"].transform(
    lambda values: values.fillna(values.median())
).fillna(overall_age_median)
clean["embarked"] = clean["embarked"].fillna(clean["embarked"].mode().iloc[0])
clean["sex"] = clean["sex"].astype("category")
clean["embarked"] = clean["embarked"].astype("category")
print("Cleaned shape:", clean.shape)
print("Missing ages after cleaning:", int(clean["age"].isna().sum()))
print("Missing embarkation ports after cleaning:", int(clean["embarked"].isna().sum()))
print("Rows retained:", len(clean), "| duplicate-looking profiles retained:", exact_duplicate_profiles)

Cleaned shape: (891, 14)
Missing ages after cleaning: 0
Missing embarkation ports after cleaning: 0
Rows retained: 891 | duplicate-looking profiles retained: 107


## 3. Compare survival by sex, passenger class, and age band

In [4]:
clean["age_band"] = pd.cut(
    clean["age"], bins=[0, 12, 18, 35, 60, 101],
    labels=["Child", "Teen", "Adult 18–34", "Adult 35–59", "60+"], right=False,
)
by_sex = clean.groupby("sex", observed=True)["survived"].agg(["mean", "count"])
by_class = clean.groupby("class", observed=True)["survived"].agg(["mean", "count"])
by_age = clean.groupby("age_band", observed=True)["survived"].agg(["mean", "count"])
display(by_sex.style.format({"mean": "{:.1%}"}).set_caption("Survival rate by sex"))
display(by_class.style.format({"mean": "{:.1%}"}).set_caption("Survival rate by passenger class"))
display(by_age.style.format({"mean": "{:.1%}"}).set_caption("Survival rate by age band"))

## Findings

1. Compare the female and male survival rates.
2. Compare first- and third-class survival rates.
3. Compare the child and adult age bands; describe the association without implying that age alone caused the outcome.

The exact rates and group sizes are printed above. These are descriptive patterns in a historical passenger sample, not causal estimates.